In [ ]:
import torch
import os
from tqdm import tqdm
import torch.optim as optim
import wandb
from torch.utils.data import DataLoader
import pickle

from modules.dataset import *
from modules.decoder import *
from modules.encoder import *
from modules.model import *
from modules.utils import *

In [ ]:
#Loading saved data
def load_data(path):
    print(f"Loading from {path}...")
    with open(path, 'rb') as f:
        data, labels = pickle.load(f)
    print("Data loaded successfully!")
    return data, labels

# Usage
load_dir = '/lambda/nfs/signlanguage/' #load directory (change as needed)

train_data, train_labels = load_data(
    os.path.join(load_dir, 'train_set.pkl')
)

val_data, val_labels = load_data(
    os.path.join(load_dir, 'val_set.pkl')
)

In [ ]:
PAD_TOKEN = "<pad>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"

# =========================
# 0. DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================
# 1. BUILD VOCAB FROM TRAIN LABELS
# ========================
max_vocab = 30000

stoi, itoi, PAD_ID, BOS_ID, EOS_ID, UNK_ID = build_limited_vocab(
    train_labels,
    max_vocab=max_vocab,
    lowercase=True  # set True if you want to lowercase everything
)

vocab_size = len(itoi)
print("Vocab size:", vocab_size)
print("PAD_ID:", PAD_ID, "BOS_ID:", BOS_ID, "EOS_ID:", EOS_ID, "UNK_ID:", UNK_ID)

loss_fn = nn.CrossEntropyLoss(
    ignore_index=PAD_ID, 
    label_smoothing=0.1  # Forces the model to be less confident about <eos>
)

# =========================
# 2. DATASETS
# =========================
train_dataset = SignDataset(
    data_dict=train_data,
    labels_dict=train_labels,
    stoi=stoi,
    BOS_ID=BOS_ID,
    EOS_ID=EOS_ID,
    UNK_ID=UNK_ID,
    lowercase=True,
)

val_dataset = SignDataset(
    data_dict=val_data,
    labels_dict=val_labels,
    stoi=stoi,          # IMPORTANT: same vocab as train
    BOS_ID=BOS_ID,
    EOS_ID=EOS_ID,
    UNK_ID=UNK_ID,
    lowercase=True,
)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

# =========================
# 3. DATALOADERS
# =========================
batch_size = 8  # change depending on GPU memory

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers = 4,
    pin_memory = True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers = 4,
    pin_memory = True
)

# =========================
# 4. BUILD GRAPH + ST-GCN ENCODER
# =========================
# Use one training sample to infer (T, J, C)
some_sid = next(iter(train_data.keys()))
sample = train_data[some_sid]

pose = to_TJC(sample["pose_keypoints_2d"])       # (T, J_body, 3)
face = to_TJC(sample["face_keypoints_2d"])       # (T, J_face, 3)
lh   = to_TJC(sample["hand_left_keypoints_2d"])  # (T, J_lh, 3)
rh   = to_TJC(sample["hand_right_keypoints_2d"]) # (T, J_rh, 3)

T0, J_body, C_in = pose.shape   # C_in should be 3 (x, y, conf)
J_face = face.shape[1]
J_lh   = lh.shape[1]
J_rh   = rh.shape[1]
V_total = J_body + J_face + J_lh + J_rh

graph_args = build_graph(V_total, edges())

stgcn_encoder = STGCNEncoder(
    in_channels=C_in,            
    graph_args=graph_args,
    edge_importance_weighting=True,
    dropout=0.5,
)

print("ST-GCN output dim:", stgcn_encoder.output_dim)

# =========================
# 5. FULL MODEL (ST-GCN ENCODER + TRANSFORMER DECODER)
# =========================
model = Sign2TextModel(
    stgcn_encoder=stgcn_encoder,
    vocab_size=vocab_size,
    d_model=stgcn_encoder.output_dim,
    dropout = 0.4
).to(device)

# =========================
# 6. OPTIMIZER
# =========================
encoder_learning_rate = 2e-4
decoder_learning_rate = 5e-5
optimizer = torch.optim.Adam(
    [
        {"params": model.encoder.parameters(), "lr": encoder_learning_rate},
        {"params": model.decoder.parameters(), "lr": decoder_learning_rate},
    ],
    betas=(0.9, 0.98),
    weight_decay=1e-4
)

print("Initialization complete.")

In [ ]:
#Parameters for training

accum_steps = 4
max_norm = 1.0
scaler = amp.GradScaler()
num_epochs = 100
global_step = 0

total_steps = (len(train_loader) * num_epochs) // accum_steps
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=2e-4,
    total_steps=total_steps,
    pct_start=0.1  # Warmup first 10%
)

In [ ]:
wandb.init(
    project="sign-language-translation",
    config={
        "batch_size": batch_size,
        "effective_batch_size": batch_size * accum_steps,
        "encoder_learning_rate": encoder_learning_rate,
        "decoder_learning_rate": decoder_learning_rate,
        "epochs": num_epochs,
    }
)
model_name = f"{wandb.run.name}"

ROOT = "./"
CHECKPOINT_DIR = os.path.join(ROOT, "checkpoints", model_name)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Saving checkpoints to: {CHECKPOINT_DIR}")

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    num_batches = len(train_loader)
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for batch_idx, (x_batch, y_batch, x_length) in enumerate(progress_bar, start=1):
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        x_length = x_length.to(device)

        # Prepare inputs and targets
        y_in  = y_batch[:, :-1]            # (N, T_in)  e.g. [<bos>, w1, ..., wT]
        y_out = y_batch[:, 1:]             # (N, T_in)  e.g. [w1, ..., wT, <eos>]
        y_in_t  = y_in.transpose(0, 1)     # (T_in, N)
        y_out_t = y_out.transpose(0, 1)    # (T_in, N)

        with amp.autocast():
            logits = model(x_batch, y_in_t, x_length)    # (T_in, N, V)

            T_tgt, N, V = logits.shape
            logits_flat  = logits.view(T_tgt * N, V)
            targets_flat = y_out_t.reshape(-1)

            loss = loss_fn(logits_flat, targets_flat)
            loss = loss / accum_steps

        # ----- 2. Scaled Backward -----
        scaler.scale(loss).backward()

        #--- 3. Optimizer Step (every accum_steps) ---
        if (batch_idx % accum_steps) == 0:
            # Unscale gradients
            scaler.unscale_(optimizer)

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)

            # Optimizer step
            scaler.step(optimizer)
            scaler.update()

            scheduler.step()
            optimizer.zero_grad()

            current_lr = scheduler.get_last_lr()[0]
            wandb.log({"lr": current_lr}, step=global_step)
            global_step += 1
        
        # ----- 4. Logging -----
        current_loss_val = loss.item() * accum_steps  # scale back to original loss
        total_loss += current_loss_val

        avg_loss_so_far = total_loss / batch_idx

        if (batch_idx % accum_steps) == 0:
            wandb.log(
                {"train/batch_loss": current_loss_val},
                step=global_step
            )

        progress_bar.set_postfix({
            "batch_loss": f"{current_loss_val:.4f}",
            "avg_loss": f"{avg_loss_so_far:.4f}"
        })

        del x_batch, y_batch, y_in, y_out, logits, logits_flat, loss

        if batch_idx % 50 == 0:
            torch.cuda.empty_cache()
   
   # ----- End of Epoch -----

    epoch_loss = total_loss / num_batches
    wandb.log(
        {"train/epoch_loss": epoch_loss},
        step=global_step
    )

    # ----- Validation -----
    model.eval()
    val_loss = 0.0
    val_progress_bar = tqdm(val_loader, desc=f"Validation: Epoch {epoch+1}/{num_epochs}")

    # Clear GPU cache before validation
    torch.cuda.empty_cache()

    with torch.no_grad():
        for val_batch_idx, (x_batch, y_batch, x_length) in enumerate(val_progress_bar):
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            x_length = x_length.to(device)
            y_t = y_batch.transpose(0, 1)

            with amp.autocast():
                logits = model(x_batch, y_t, x_length)
                T_tgt, N, Vocab = logits.shape
                logits_flat  = logits.view(T_tgt * N, Vocab)
                targets_flat = y_t.reshape(-1)
                loss = loss_fn(logits_flat, targets_flat)
                
            val_loss += loss.item()

            if val_batch_idx == 0:
                # Decode prediction
                # Use your greedy_decode_forced function here
                pred_words = greedy_decode(model, x_batch[0:1], x_length, itoi, BOS_ID, EOS_ID, PAD_ID, UNK_ID)
                pred_sent = " ".join(pred_words)
                
                # Decode truth
                true_ids = y_t[:, 0]
                true_words = decode_ids_to_words(true_ids, itoi, PAD_ID, BOS_ID, EOS_ID)
                true_sent = " ".join(true_words)
                
                wandb.log({
                    "val/prediction": pred_sent,
                    "val/truth": true_sent
                }, step=global_step)
            
            #Clear variables
            del x_batch, y_batch, logits, loss
            
    val_loss /= len(val_loader)
    wandb.log(
        {"val/epoch_loss": val_loss},
        step=global_step
    )
    print(f"==> Epoch {epoch+1}: loss = {epoch_loss:.4f} | val_loss = {val_loss:.4f}")

    #CHECKPOINT
    checkpoint = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "stoi": stoi,
        "PAD_ID": PAD_ID,
        "BOS_ID": BOS_ID,
        "EOS_ID": EOS_ID,
        "UNK_ID": UNK_ID,
    }

    save_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_epoch_{epoch}.pt")
    torch.save(checkpoint, save_path)
    print(f"Saved checkpoint: {save_path}")
